# Notebook 07: Dimensión de Puntualidad

**Duración**: 30 minutos | **Nivel**: Principiante

## ¿Qué es Puntualidad?

**Puntualidad** asegura que los datos estén actualizados y disponibles cuando se necesitan.

### Impacto:
- Decisiones basadas en datos obsoletos
- SLAs incumplidos
- Datos futuros (errores de sistema)

In [ ]:
import great_expectations as gx
import pandas as pd
from datetime import datetime, timedelta

df = pd.read_csv("../data/ventas_sucias.csv")
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

print(f"Fecha más antigua: {df['order_date'].min()}")
print(f"Fecha más reciente: {df['order_date'].max()}")
print(f"Fecha actual: {datetime.now().date()}")

## Detectar Fechas Futuras

In [ ]:
fechas_futuras = df[df['order_date'] > pd.Timestamp.now()]
print(f"Registros con fechas futuras: {len(fechas_futuras)}")

if len(fechas_futuras) > 0:
    print("\nEjemplos:")
    print(fechas_futuras[['order_id', 'order_date']].head())

In [ ]:
# Configurar
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Suite
suite = context.suites.add(gx.ExpectationSuite(name="puntualidad"))

# Fechas no deben ser futuras
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_date",
        min_value="2020-01-01",
        max_value=datetime.now().strftime("%Y-%m-%d"),
        meta={"dimension": "Puntualidad"}
    )
)

suite.save()

val_def = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite, name="val_puntualidad")
)

resultado = val_def.run(batch_parameters={"dataframe": df})
print(f"\n¿Validación exitosa?: {' SÍ' if resultado.success else ' NO'}")

## Validar Frescura de Datos

In [ ]:
# Datos deben ser de los últimos 30 días
hace_30_dias = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
datos_recientes = df[df['order_date'] >= hace_30_dias]

print(f"Registros de los últimos 30 días: {len(datos_recientes)}")
print(f"Porcentaje: {len(datos_recientes)/len(df)*100:.1f}%")

##  Ejercicio

Crea una validación que asegure que al menos el 80% de los datos sean de los últimos 7 días.

In [ ]:
# TU CÓDIGO AQUÍ
pass

In [ ]:
context.build_data_docs()
context.open_data_docs()

##  Resumen

1.  Puntualidad valida que datos estén actualizados
2.  Detecta fechas futuras (errores de sistema)
3.  Valida frescura de datos

